# Plate 03 · Detector + Corners Training (YOLO11n-pose)

**Architecture:** YOLO11n-**pose** (≈2.9 M params), 640 px input, 5 classes (plate type) + **4 corner keypoints**.
The corners let the app warp every plate to the canonical Kuwaiti layout, so the recogniser only reads fixed cells.
Why: best accuracy/latency on phones in its size class, built-in export to ONNX / TFLite int8 / Core ML.
Licence: Ultralytics is **AGPL-3.0** – a closed-source commercial app needs an Ultralytics Enterprise licence;
the Apache-2.0 fallback is **RTMPose / RTMDet-tiny** (box) + a small corner-regression head.

Recipe: pre-train boxes on Open Images plates → train pose on synthetic Kuwaiti scenes (notebook 02)
→ fine-tune on real Kuwaiti pilot data. Targets **box mAP50 ≥ 0.97**, mean corner error ≤ 3 % of plate width.

In [ ]:
# --- Setup: make Common_Utils importable (works locally, on Kaggle and on Colab) ---
import sys
from pathlib import Path

# On Kaggle/Colab first clone the repo, e.g.:
# !git clone https://github.com/AhmedADesoky/Car_Services_App.git && %cd Car_Services_App/AI_Models/Plate_Recognition
AI_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(AI_ROOT))

from Common_Utils.Paths import DATA, RUNS, WEIGHTS, model_dir
from Common_Utils.Data_Utils import seed_everything
seed_everything(42)
print("Data:", DATA, "| Runs:", RUNS, "| Weights:", WEIGHTS)

In [ ]:
from ultralytics import YOLO

DATA_YAML = DATA / "Plates" / "Detection" / "plates.yaml"     # kpt_shape: [4, 3] (notebook 01)
model = YOLO("yolo11n-pose.pt")     # COCO-pose pretrained starting point
results = model.train(
    data=str(DATA_YAML),
    imgsz=640, epochs=150, batch=32, patience=30,
    optimizer="AdamW", lr0=0.002, cos_lr=True,
    mosaic=1.0, close_mosaic=15, mixup=0.1, degrees=5, perspective=0.0005,
    fliplr=0.0,                        # plates are not mirror-symmetric: no horizontal flips
    hsv_v=0.5,                         # strong brightness changes: day / night bays, Kuwaiti summer glare
    pose=12.0,                         # weight of the corner loss
    project=str(RUNS / "Plate_Detector"), name="yolo11n_pose_kw_v1", exist_ok=True,
)

In [ ]:
best = RUNS / "Plate_Detector" / "yolo11n_pose_kw_v1" / "weights" / "best.pt"
metrics = YOLO(str(best)).val(data=str(DATA_YAML), split="test", imgsz=640)
print("box mAP50:", metrics.box.map50, "| pose mAP50:", metrics.pose.map50)

In [ ]:
# Quick visual check on a few test images
import random
from Common_Utils.Data_Utils import list_images
samples = random.sample(list_images(DATA / "Plates" / "Detection" / "images" / "test"), k=4)
YOLO(str(best)).predict([str(s) for s in samples], save=True, project=str(RUNS / "Plate_Detector"), name="preview")